# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asraserver06/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
%pip install -q duckdb huggingface_hub pandas

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [20]:
# Grain check: no duplicate report_date x client x content combos
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").show()

# Window check: does this partition really cover only March 2026?
con.sql("""
SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) AS row_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

┌────────────┬────────────┬───────────┐
│  min_date  │  max_date  │ row_count │
│    date    │    date    │   int64   │
├────────────┼────────────┼───────────┤
│ 2026-03-01 │ 2026-03-31 │   9841378 │
└────────────┴────────────┴───────────┘



In [21]:
result = con.sql("""
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
LIMIT 1
""").df()

for col in result['column_name']:
    print(col)

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


**Unit of analysis:** One row = one content page, on one specific report_date, for one client — grain is report_date × client_hash_id × content_hash_id. This comes from `fact_content_daily_performance` in the FlyRank warehouse.

**Time window:** The full table spans 2025-01-27 to 2026-06-30 (~17 months), partitioned by month. This contract uses the mid-panel month `month=2026-03` to develop and verify — not `_sample` (June 2026, the final/sealed month), which is reserved as the test window for label evaluation.

In [22]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}');")

con.sql("""
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
LIMIT 1
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [23]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [25]:
gsc_pct = 3611061 / 9841378 * 100
ga4_pct = 413966 / 9841378 * 100
print(f"GSC available: {gsc_pct:.1f}% of rows")
print(f"GA4 available: {ga4_pct:.1f}% of rows")

GSC available: 36.7% of rows
GA4 available: 4.2% of rows


In [26]:
con.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



**Fields: feature / label / context / excluded**

- **Feature (knowable before the decision moment):**
  - `gsc_impressions` — search visibility, observed daily
  - `gsc_clicks` — actual clicks, observed daily
  - `gsc_avg_position` — average search rank that day
  - `ga4_engaged_sessions` — how many sessions showed real engagement
  - `scroll_events` — a proxy for on-page engagement depth

- **Label / proxy (what we're predicting — never also a feature):**
  - A declining-content proxy built from `gsc_clicks` and `gsc_impressions` trend across the days in March 2026 (e.g. comparing early-March vs late-March click rate for the same content). Not yet finalized as a strict formula — that's next week's job — but no column that touches this trend may also appear as a feature, to avoid leakage.

- **Context (for grouping/joining only, never learned from):**
  - `client_hash_id`, `content_hash_id` — pseudonymous IDs, used only to group rows into one page's history
  - `report_date` — used to define the time window, not as a raw model input
  - `month` — partition marker only

- **Excluded (with reason):**
  - `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — these are availability/coverage flags, not content-performance signals; useful only for filtering rows in, not for the model to learn from
  - `sessions_paid`, `sessions_social`, `sessions_referral`, `sessions_direct`, `ai_chatgpt`/`ai_perplexity`/etc. — excluded for this lane because they reflect traffic-source mix, not the page's own search performance, and adding all of them would blur the "declining in search" signal we actually care about
  - `gsc_sum_position` — excluded because it's redundant with `gsc_avg_position` (sum vs. average of the same underlying values) and could double-count the same signal if both were included

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
con.sql("""
CREATE OR REPLACE TABLE march_agg AS
SELECT
  content_hash_id,
  client_hash_id,
  SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_early,
  SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_early,
  AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position END) AS avg_position_early,
  COUNT(CASE WHEN report_date <= '2026-03-15' AND gsc_impressions > 0 THEN 1 END) AS active_days_early,
  SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_late,
  SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_late
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE gsc_data_available IS TRUE
GROUP BY content_hash_id, client_hash_id
HAVING impressions_early > 0 AND impressions_late > 0
""")

con.sql("SELECT COUNT(*) FROM march_agg").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       141467 │
└──────────────┘



In [28]:
con.sql("""
CREATE OR REPLACE TABLE feature_frame AS
SELECT
  content_hash_id,
  client_hash_id,
  impressions_early,
  clicks_early,
  CAST(clicks_early AS FLOAT) / NULLIF(impressions_early, 0) AS ctr_early,
  avg_position_early,
  active_days_early,
  CAST(clicks_late AS FLOAT) / NULLIF(impressions_late, 0) AS ctr_late,
  CASE WHEN (CAST(clicks_late AS FLOAT) / NULLIF(impressions_late,0))
           < (CAST(clicks_early AS FLOAT) / NULLIF(impressions_early,0))
       THEN 1 ELSE 0 END AS is_declining_label
FROM march_agg
""")

con.sql("SELECT is_declining_label, COUNT(*) FROM feature_frame GROUP BY is_declining_label").show()

┌────────────────────┬──────────────┐
│ is_declining_label │ count_star() │
│       int32        │    int64     │
├────────────────────┼──────────────┤
│                  0 │       104748 │
│                  1 │        36719 │
└────────────────────┴──────────────┘



**The leakage trap:** I deliberately added `ctr_diff_leaked` (the difference between late-March CTR and early-March CTR) as a feature — this is directly derived from the same values used to define `is_declining_label` (`ctr_late < ctr_early`). ROC-AUC moved from an honest 0.890 to 0.919 with this leak included. The jump was real but smaller than expected for such a direct leak, likely because logistic regression is a linear model and doesn't perfectly exploit a comparison-based signal even when it's present — a reminder that "the leak didn't blow up the score" doesn't mean there's no leak, it means the model type mattered. I removed `ctr_diff_leaked` (and all `*_late` columns) and kept only the honest, pre-decision-moment feature set: `impressions_early`, `clicks_early`, `ctr_early`, `avg_position_early`, `active_days_early`, giving a final honest ROC-AUC of 0.890.

In [29]:
# Trap removed — dropping label-derived columns, keeping only pre-decision-moment features
final_features = honest_features  # impressions_early, clicks_early, ctr_early, avg_position_early, active_days_early
print("Final honest ROC-AUC (leak removed):", honest_score)


Final honest ROC-AUC (leak removed): 0.8898866931298149


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [32]:
print(f"GSC available: {gsc_pct:.1f}% of rows")
print(f"GA4 available: {ga4_pct:.1f}% of rows")

GSC available: 36.7% of rows
GA4 available: 4.2% of rows


**Data limits — what this data can never tell you:**

1. **Most rows lack real engagement data.** Only 36.7% of rows in this month have `gsc_data_available = TRUE`, and just 4.2% have `ga4_data_available = TRUE`. This means our features (`clicks_early`, `ctr_early`, etc.) are only meaningful for about a third of all pages — the rest have no search signal to build from at all, and GA4-based engagement (`ga4_engaged_sessions`, `scroll_events`) is unusable for this lane at anything beyond a small fraction of pages.

2. **Client history is unbalanced.** Per the warehouse's panel warning, `gsc_data_start` and `ga4_data_start` differ wildly per client — some clients have far more history than others in this same March window, which means our early/late split isn't equally reliable across every client.

3. **A within-month CTR comparison is a short-term proxy, not a true "declining" signal.** Two weeks of data can reflect normal week-to-week noise (seasonality, one bad week) rather than a real, sustained decline. A more defensible label would use a longer observation window across multiple months — which this contract intentionally does not attempt yet.

4. **This data cannot explain *why* a page is declining** — only whether clicks/CTR moved. Causes (algorithm changes, competitor content, seasonal demand shifts) aren't observable in this table at all.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.